# Lab 06 - Linear Regression: Impact of AI on Students

        Source dataset: `Datasets/Impact of AI on Students/ai_student_impact_dataset.csv`

        This notebook adapts the class lab pattern to the student-impact dataset. The source file is never modified.

        ## Lab concepts used

        - Compare ordinary and regularised linear models.
- Predict skill retention and GPA change.
- Add polynomial terms only to a controlled numeric subset.

        Interpretation is predictive and associative only. The Kaggle source does not document how the records were collected or whether they represent observed students.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

def find_dataset():
    relative = Path("Datasets/Impact of AI on Students/ai_student_impact_dataset.csv")
    for start in [Path.cwd(), *Path.cwd().parents]:
        candidate = start / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {relative} from {Path.cwd()}")

DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns from {DATA_PATH}")

In [ ]:
IDENTIFIER = "Student_ID"
OUTCOMES = ["Post_Semester_GPA", "Skill_Retention_Score", "Burnout_Risk_Level"]
EARLY_RISK_FEATURES = [
    "Major_Category", "Year_of_Study", "Pre_Semester_GPA",
    "Weekly_GenAI_Hours", "Primary_Use_Case",
    "Prompt_Engineering_Skill", "Tool_Diversity", "Paid_Subscription",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Institutional_Policy",
]
EXPANDED_FEATURES = EARLY_RISK_FEATURES + ["Anxiety_Level_During_Exams"]

df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]
df["GPA_Declined"] = (df["GPA_Change"] < 0).astype(int)

assert IDENTIFIER not in EARLY_RISK_FEATURES
assert not set(OUTCOMES).intersection(EARLY_RISK_FEATURES)
print("Leakage policy ready. Primary burnout model excludes anxiety and all post-semester outcomes.")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def make_preprocessor(frame, scale_numeric=True):
    categorical = [
        column for column in frame.columns
        if pd.api.types.is_string_dtype(frame[column])
        or pd.api.types.is_bool_dtype(frame[column])
    ]
    numeric = [column for column in frame.columns if column not in categorical]
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                categorical,
            ),
        ]
    )

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def compare_regressors(target):
    X = df[EXPANDED_FEATURES]
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE
    )
    models = {
        "Linear": LinearRegression(),
        "Ridge": Ridge(alpha=1.0),
        "Lasso": Lasso(alpha=0.01, max_iter=5000),
        "Elastic Net": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000),
    }
    rows = []
    for name, model in models.items():
        pipe = Pipeline([
            ("preprocess", make_preprocessor(X_train)),
            ("model", model),
        ])
        pipe.fit(X_train, y_train)
        prediction = pipe.predict(X_test)
        rows.append({
            "target": target,
            "model": name,
            "MAE": mean_absolute_error(y_test, prediction),
            "RMSE": mean_squared_error(y_test, prediction) ** 0.5,
            "R2": r2_score(y_test, prediction),
        })
    return pd.DataFrame(rows)

display(pd.concat([
    compare_regressors("Skill_Retention_Score"),
    compare_regressors("GPA_Change"),
], ignore_index=True))

## Controlled polynomial Ridge experiment

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

numeric_subset = [
    "Pre_Semester_GPA", "Weekly_GenAI_Hours",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Tool_Diversity",
]
X = df[numeric_subset]
y = df["GPA_Change"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
polynomial_ridge = Pipeline([
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
    ("ridge", Ridge(alpha=10.0)),
])
polynomial_ridge.fit(X_train, y_train)
prediction = polynomial_ridge.predict(X_test)
print({
    "MAE": mean_absolute_error(y_test, prediction),
    "RMSE": mean_squared_error(y_test, prediction) ** 0.5,
    "R2": r2_score(y_test, prediction),
})

## What was learned from Lab 6

Regularised linear models establish transparent baselines. GPA change is preferable to raw post-GPA because it reduces the dominance of the baseline GPA while retaining interpretable academic change.